<a href="https://colab.research.google.com/github/alexandrelombard/ai54-notebooks/blob/master/04_RNN_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd

import nltk
from nltk.tokenize import word_tokenize


import matplotlib.pyplot as plt

import random

nltk.download('punkt', quiet=True)

True

In [ ]:
# Prepare the data
with open('romeo_and_juliet.txt') as file:
    content = file.read()

content = content.lower()
tokens = word_tokenize(content)

vocabulary = list(set(tokens))

word2idx = {w: i for i, w in enumerate(vocabulary)}
idx2word = {i: w for w, i in word2idx.items()}

print(tokens[0:100])

['1595', 'the', 'tragedy', 'of', 'romeo', 'and', 'juliet', 'by', 'william', 'shakespeare', 'dramatis', 'personae', 'chorus', '.', 'escalus', ',', 'prince', 'of', 'verona', '.', 'paris', ',', 'a', 'young', 'count', ',', 'kinsman', 'to', 'the', 'prince', '.', 'montague', ',', 'heads', 'of', 'two', 'houses', 'at', 'variance', 'with', 'each', 'other', '.', 'capulet', ',', 'heads', 'of', 'two', 'houses', 'at', 'variance', 'with', 'each', 'other', '.', 'an', 'old', 'man', ',', 'of', 'the', 'capulet', 'family', '.', 'romeo', ',', 'son', 'to', 'montague', '.', 'tybalt', ',', 'nephew', 'to', 'lady', 'capulet', '.', 'mercutio', ',', 'kinsman', 'to', 'the', 'prince', 'and', 'friend', 'to', 'romeo', '.', 'benvolio', ',', 'nephew', 'to', 'montague', ',', 'and', 'friend', 'to', 'romeo', 'tybalt', ',']


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# Build a simple model
class Predict(nn.Module):
    def __init__(self, vocabulary_size, embedding_dim, hidden_dim):
        super(Predict, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocabulary_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, vocabulary_size)

    def forward(self, x):
        x = self.embedding(x)   # (L, N, In) --> (L, N, E)
        x, _ = self.lstm(x)     # (L, N, E) --> (L, N, H)
        x = self.linear(x)      # (L, N, H) --> (L, N, Out)
        return x

predict = Predict(len(vocabulary), 256, 256).to(device)

In [ ]:
# Build training data: predict next word given previous words (n-gram sequences)
context_size = 5  # number of previous words used to predict the next one
sequences = []
targets = []
# Create rolling windows over the token sequence
for i in range(context_size, len(tokens)):
    context = tokens[i - context_size:i]
    target = tokens[i]
    # Skip tokens not in mapping just in case
    if all(w in word2idx for w in context) and target in word2idx:
        sequences.append([word2idx[w] for w in context])
        targets.append(word2idx[target])

# Convert to tensors with shape (seq_len, batch) expected by nn.LSTM default
# We'll transpose later inside the loop to feed as (seq_len, batch)
X = torch.tensor(sequences, dtype=torch.long, device=device)  # (N, context_size)
y = torch.tensor(targets, dtype=torch.long, device=device)    # (N,)

In [ ]:
# Simple training loop
optimizer = torch.optim.Adam(predict.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

batch_size = 128
epochs = 200  # keep small; adjust as needed

num_batches = (X.size(0) + batch_size - 1) // batch_size
for epoch in range(epochs):
    perm = torch.randperm(X.size(0), device=device)
    X = X[perm]
    y = y[perm]
    epoch_loss = 0.0
    for b in range(num_batches):
        start = b * batch_size
        end = min((b + 1) * batch_size, X.size(0))
        xb = X[start:end]  # (B, context)
        yb = y[start:end]  # (B)

        # Prepare for LSTM: (seq_len, batch)
        xb_seq = xb.t().contiguous()  # (context, B)

        optimizer.zero_grad()
        logits = predict(xb_seq)  # (context, B, vocab)
        # Use only the last time step to predict next word
        logits_last = logits[-1]  # (B, vocab)
        loss = criterion(logits_last, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * (end - start)

    print(f"Epoch {epoch+1}/{epochs} - loss: {epoch_loss / X.size(0):.4f}")

Epoch 1/100 - loss: 0.0300
Epoch 2/100 - loss: 0.0285
Epoch 3/100 - loss: 0.0262
Epoch 4/100 - loss: 0.0256
Epoch 5/100 - loss: 0.0241
Epoch 6/100 - loss: 0.0256
Epoch 7/100 - loss: 0.0284
Epoch 8/100 - loss: 0.0284
Epoch 9/100 - loss: 0.0298
Epoch 10/100 - loss: 0.0265
Epoch 11/100 - loss: 0.0232
Epoch 12/100 - loss: 0.0209
Epoch 13/100 - loss: 0.0199
Epoch 14/100 - loss: 0.0198
Epoch 15/100 - loss: 0.0203
Epoch 16/100 - loss: 0.0199
Epoch 17/100 - loss: 0.0199
Epoch 18/100 - loss: 0.0202
Epoch 19/100 - loss: 0.0204
Epoch 20/100 - loss: 0.0211
Epoch 21/100 - loss: 0.0580
Epoch 22/100 - loss: 0.1404
Epoch 23/100 - loss: 0.0396
Epoch 24/100 - loss: 0.0239
Epoch 25/100 - loss: 0.0218
Epoch 26/100 - loss: 0.0208
Epoch 27/100 - loss: 0.0204
Epoch 28/100 - loss: 0.0201
Epoch 29/100 - loss: 0.0201
Epoch 30/100 - loss: 0.0200
Epoch 31/100 - loss: 0.0200
Epoch 32/100 - loss: 0.0202
Epoch 33/100 - loss: 0.0204
Epoch 34/100 - loss: 0.0200
Epoch 35/100 - loss: 0.0200
Epoch 36/100 - loss: 0.0195
E

In [ ]:
# Quick test: given a random context, generate next 20 words
with torch.no_grad():
    # pick a random starting point
    start_idx = random.randint(context_size, len(tokens) - 1)
    context = tokens[start_idx - context_size - 2:start_idx]
    print("Seed:", " ".join(context))

    generated = []
    for _ in range(20):
        context_idxs = torch.tensor([[word2idx[w] for w in context]], device=device)
        context_idxs = context_idxs.t()  # (context, 1)
        logits = predict(context_idxs)
        next_token_logits = logits[-1, 0]
        next_idx = int(torch.argmax(next_token_logits))
        next_word = idx2word[next_idx]
        generated.append(next_word)
        context = context[1:] + [next_word]

    print("Generated:", " ".join(generated))
    print("Expected:", " ".join(tokens[start_idx:start_idx + 20]))

Seed: prince . i can discover all the
Generated: unlucky manage of this fatal . hence . the heavens that one of our streets , by my ears !
Expected: unlucky manage of this fatal brawl . there lies the man , slain by young romeo , that slew thy


In [ ]:
# Quick test: with a manual context, generate next 20 words
with torch.no_grad():
    # pick a random starting point
    context = ['how', 'is', 'it']
    print("Seed:", " ".join(context))

    generated = []
    for _ in range(20):
        context_idxs = torch.tensor([[word2idx[w] for w in context]], device=device)
        context_idxs = context_idxs.t()  # (context, 1)
        logits = predict(context_idxs)
        next_token_logits = logits[-1, 0]
        next_idx = int(torch.argmax(next_token_logits))
        next_word = idx2word[next_idx]
        generated.append(next_word)
        context = context[1:] + [next_word]

    print("Generated:", " ".join(generated))

Seed: how is it
Generated: now , by my letters know the language . serv . if you be gone , sir , so unluckily
